In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# File paths
stock_data1 = r"D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_1.xlsx"
stock_data2 = r"D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_2.xlsx"
rf_path = r"D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\weekly_risk_free_rate.xlsx"

# Load stock return data (assuming columns: Stkcd, Trdwnt, Wretnd)
weekly_returns1 = pd.read_excel(stock_data1, dtype={'Stkcd': str})
weekly_returns2 = pd.read_excel(stock_data2, dtype={'Stkcd': str})
weekly_returns = pd.concat([weekly_returns1, weekly_returns2], ignore_index=True)
weekly_returns = weekly_returns[weekly_returns['Trdwnt'] < '2023-01'].reset_index(drop=True)

# Load risk-free return data
df = pd.read_excel(rf_path, sheet_name='Sheet1')
df['trading_date_yw'] = pd.to_datetime(df['trading_date_yw'])
df.set_index('trading_date_yw', inplace=True)
df['Trdwnt'] = df.index.to_period('W').strftime('%Y-%U')

# Merge stock and risk-free data
final_df = pd.merge(weekly_returns, df, on='Trdwnt', how='left')

# Calculate market returns
market_returns = final_df.groupby('Trdwnt')['Wretnd'].mean().reset_index()
market_returns.rename(columns={'Wretnd': 'Market_Return'}, inplace=True)
final_df = pd.merge(final_df, market_returns, on='Trdwnt', how='left')

# Sort by trading week
final_df.sort_values(by=['Trdwnt'], inplace=True)

# Display basic info
print("First few rows of merged DataFrame:")
print(final_df.head())

First few rows of merged DataFrame:
         Stkcd   Trdwnt    Wretnd  risk_free_return Market_Return
0       000001  2017-01  0.003297          0.000571      0.016611
13592   000062  2017-01 -0.029567          0.000571      0.016611
429664  300016  2017-01  -0.00379          0.000571      0.016611
818649  600555  2017-01    0.0064          0.000571      0.016611
429966  300017  2017-01  0.020705          0.000571      0.016611


# Temporal Filtering

In [7]:
# Split into three 33-week periods
sorted_weeks = sorted(final_df['Trdwnt'].unique())
periods = [sorted_weeks[i*33:(i+1)*33] for i in range(3)]

# Filter stocks present in all periods
common_stocks = set.intersection(*[
    set(final_df[final_df['Trdwnt'].isin(p)]['Stkcd'].unique()) 
    for p in periods
])
final_df = final_df[final_df['Stkcd'].isin(common_stocks)]

# Phase 1: Individual Beta Calculation

In [8]:
def calculate_beta(df):
    """Calculate beta using time-series regression"""
    X = sm.add_constant(df['Market_Return'])
    model = sm.OLS(df['Wretnd'], X, missing='drop').fit()
    return model.params['Market_Return']

phase1_data = final_df[final_df['Trdwnt'].isin(periods[0])]
betas = phase1_data.groupby('Stkcd').apply(calculate_beta).to_frame('beta')

# Portfolio Formation ------------------------------------------------------
betas['portfolio'] = pd.qcut(betas['beta'], q=10, labels=range(1,11))

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).